# Simple — новые mesh-признаки

## 1. Данные и разбиение

In [1]:
%pip install -q catboost iterative-stratification tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'data' / 'train.csv').exists())
SIMPLE_DIR = ROOT / 'labels' / 'simple'
ARCHIVE_DIR = ROOT / 'archive'
TRAIN_DIR = ROOT / 'data' / 'train'
train = pd.read_csv(ROOT / 'data' / 'train.csv')
target_columns = train.columns.drop('item_id').tolist()

splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_index, _ = next(splitter.split(train, train[target_columns]))
train_val_df = train.iloc[train_val_index].reset_index(drop=True)

splitter = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1765, random_state=43)
train_index, val_index = next(splitter.split(train_val_df, train_val_df[target_columns]))
train_df = train_val_df.iloc[train_index].reset_index(drop=True)
val_df = train_val_df.iloc[val_index].reset_index(drop=True)

print('Train:', len(train_df))
print('Validation:', len(val_df))

c:\Users\gasre\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 6259
Validation: 1361


## 2. Признаки формы

In [3]:
def extract_shape_features(item_id):
    with np.load(TRAIN_DIR / f'{item_id}.npz') as mesh:
        vertices = mesh['vertices']

    count = min(len(vertices), 2048)
    rng = np.random.default_rng(int(item_id[:8], 16))
    indices = rng.choice(len(vertices), count, replace=False)
    points = vertices[indices].astype(np.float64)

    size = np.linalg.norm(np.ptp(points, axis=0))
    centered = (points - points.mean(axis=0)) / max(size, 1e-12)
    covariance = centered.T @ centered / count
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    eigenvalues = np.maximum(eigenvalues, 0)

    radius = np.linalg.norm(centered, axis=1)
    axis = eigenvectors[:, -1]
    axial = centered @ axis
    cylinder_radius = np.linalg.norm(centered - axial[:, None] * axis, axis=1)

    return {
        'pca_flatness': eigenvalues[0] / (eigenvalues.sum() + 1e-12),
        'pca_linearity': (eigenvalues[2] - eigenvalues[1]) / (eigenvalues[2] + 1e-12),
        'sphere_radius_cv': radius.std() / (radius.mean() + 1e-12),
        'cylinder_radius_cv': cylinder_radius.std() / (cylinder_radius.mean() + 1e-12)
    }

In [4]:
shape_columns = ['pca_flatness', 'pca_linearity', 'sphere_radius_cv', 'cylinder_radius_cv']
features_path = SIMPLE_DIR / 'simple_mesh_features.csv'

if features_path.exists():
    shape_features = pd.read_csv(features_path)
    print('Загружены готовые признаки')
else:
    rows = []
    for item_id in tqdm(train_val_df['item_id'], desc='Mesh-признаки'):
        rows.append({'item_id': item_id, **extract_shape_features(item_id)})
    shape_features = pd.DataFrame(rows)
    shape_features.to_csv(features_path, index=False)
    print('Признаки сохранены:', features_path)

assert shape_features['item_id'].is_unique
assert set(shape_features['item_id']) == set(train_val_df['item_id'])
assert np.isfinite(shape_features[shape_columns].to_numpy()).all()
shape_features.head()

Загружены готовые признаки


,item_id,pca_flatness,pca_linearity,sphere_radius_cv,cylinder_radius_cv
0,0001be62-bf9c-4c15-8a09-dce209be9715,0.125606,0.692244,0.319536,0.463382
1,0024dc8f-a2d7-4556-94dc-6b4b0f46ca0d,0.008450,0.487082,1.150275,1.061023
2,002b097b-4237-4610-952c-6095d48fb7a0,0.006855,0.251854,0.220746,0.654684
3,003f01c8-d5e8-41f7-afdf-12a1bdaa1f83,0.000071,0.908577,0.398863,0.573982
4,003fb77f-53a7-44bd-ac96-25b2cbc47f43,0.146380,0.389182,0.345032,0.524331


## 3. CatBoost с новыми признаками

In [5]:
with open(ARCHIVE_DIR / 'artifacts' / 'catboost_features_config.json', encoding='utf-8') as file:
    config = json.load(file)
assert config['target_columns'] == target_columns

old_columns = config['feature_columns']
old_features = pd.read_csv(
    ARCHIVE_DIR / 'artifacts' / 'train_mesh_features_relative.csv'
).set_index('item_id')
new_features = shape_features.set_index('item_id')[shape_columns]

X_train_old = old_features.loc[train_df['item_id'], old_columns]
X_val_old = old_features.loc[val_df['item_id'], old_columns]
X_train = X_train_old.join(new_features.loc[train_df['item_id']])
X_val = X_val_old.join(new_features.loc[val_df['item_id']])
assert X_train.index.tolist() == train_df['item_id'].tolist()
assert X_val.index.tolist() == val_df['item_id'].tolist()
assert np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_val.to_numpy()).all()

model = CatBoostClassifier(**config['parameters'])
model.fit(
    X_train, train_df['simple'],
    eval_set=(X_val, val_df['simple']),
    early_stopping_rounds=50,
    verbose=False
)

model_dir = SIMPLE_DIR / 'models'
model_dir.mkdir(exist_ok=True)
model_path = model_dir / 'simple_shape_catboost.cbm'
model.save_model(str(model_path))
print('Новых признаков:', len(shape_columns))
print('Всего признаков:', X_train.shape[1])
print('Деревьев:', model.tree_count_)
print('Модель сохранена:', model_path)

Новых признаков: 4
Всего признаков: 48
Деревьев: 402
Модель сохранена: d:\asg\СОРЕВНОВАНИЯ\AI challenge main\labels\simple\models\simple_shape_catboost.cbm


## 4. Сравнение с прежним CatBoost

In [6]:
old_model = CatBoostClassifier()
old_model.load_model(str(ARCHIVE_DIR / 'models' / 'catboost_features' / 'simple.cbm'))
assert old_model.feature_names_ == old_columns

y_val = val_df['simple'].to_numpy()
old_probabilities = old_model.predict_proba(X_val_old)[:, 1]
new_probabilities = model.predict_proba(X_val)[:, 1]
old_threshold = config['thresholds'][target_columns.index('simple')]

grid = np.arange(0.05, 0.81, 0.01)
scores = [f1_score(y_val, new_probabilities >= value) for value in grid]
new_threshold = float(grid[np.argmax(scores)])

rows = []
for name, probabilities, value in [
    ('Прежний CatBoost', old_probabilities, old_threshold),
    ('Новые признаки, старый порог', new_probabilities, old_threshold),
    ('Новые признаки, новый порог', new_probabilities, new_threshold)
]:
    predicted = probabilities >= value
    rows.append({
        'Вариант': name,
        'Порог': value,
        'Precision': precision_score(y_val, predicted),
        'Recall': recall_score(y_val, predicted),
        'F1': f1_score(y_val, predicted),
        'AP': average_precision_score(y_val, probabilities),
        'FP': int(((y_val == 0) & predicted).sum()),
        'FN': int(((y_val == 1) & ~predicted).sum())
    })

results = pd.DataFrame(rows).round(4)
results

,Вариант,Порог,Precision,Recall,F1,AP,FP,FN
0,Прежний CatBoost,0.25,0.5964,0.7924,0.6805,0.7308,155,60
1,"Новые признаки, старый порог",0.25,0.5914,0.8062,0.6823,0.7501,161,56
2,"Новые признаки, новый порог",0.26,0.5949,0.8028,0.6834,0.7501,158,57


## 5. Пропуски по группам

In [7]:
other_columns = [name for name in target_columns if name not in ['simple', 'quality']]
mixed = val_df[other_columns].sum(axis=1).gt(0).to_numpy()
groups = []

for group_name, mask in [('Только simple', ~mixed), ('Simple + другие', mixed)]:
    selected = mask & (y_val == 1)
    for model_name, probabilities, value in [
        ('Прежний', old_probabilities, old_threshold),
        ('Новый', new_probabilities, new_threshold)
    ]:
        groups.append({
            'Группа': group_name,
            'Модель': model_name,
            'Объектов': int(selected.sum()),
            'Пропущено': int((selected & (probabilities < value)).sum())
        })

pd.DataFrame(groups)

,Группа,Модель,Объектов,Пропущено
0,Только simple,Прежний,210,30
1,Только simple,Новый,210,30
2,Simple + другие,Прежний,79,30
3,Simple + другие,Новый,79,27


## 6. Значимость новых признаков

In [8]:
importance = pd.Series(model.get_feature_importance(), index=X_train.columns)
importance.sort_values(ascending=False).head(10).round(2).to_frame('Значимость')

,Значимость
min_to_max,9.05
pca_flatness,7.78
faces_per_vertex,6.04
bbox_fill,5.54
surface_area_relative,4.06
radius_mean_relative,3.42
edge_variation,3.09
radius_max_relative,3.02
edge_max_relative,2.94
edge_mean_relative,2.68


In [9]:
importance[shape_columns].sort_values(ascending=False).round(2).to_frame('Значимость')

,Значимость
pca_flatness,7.78
cylinder_radius_cv,2.11
sphere_radius_cv,1.90
pca_linearity,1.46
